In [1]:
import torch
import torch.nn as nn
import timm

print("Libraries Imported Successfully")

C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries Imported Successfully


In [2]:
class CNNModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        self.feature_layer = nn.Sequential(

            nn.Linear(feature_dim,512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512,128),
            nn.ReLU()

        )

    def forward(self,x):

        x = self.backbone(x)

        features = self.feature_layer(x)

        return features

In [3]:
class ClinicalModel(nn.Module):

    def __init__(self,input_dim=5):

        super().__init__()

        self.feature_layer = nn.Sequential(

            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64,32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),

            nn.Linear(32,16),
            nn.ReLU()

        )

    def forward(self,x):

        return self.feature_layer(x)

In [4]:
class GeoModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(1,8),
            nn.ReLU(),

            nn.Linear(8,8),
            nn.ReLU()

        )

    def forward(self,x):

        return self.network(x)

In [5]:
class FusionNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.image_model = CNNModel()

        self.clinical_model = ClinicalModel()

        self.geo_model = GeoModel()

        self.classifier = nn.Sequential(

            nn.Linear(128+16+8,128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64,2)

        )

    def forward(self,image,clinical,geo):

        image_features = self.image_model(image)

        clinical_features = self.clinical_model(clinical)

        geo_features = self.geo_model(geo)

        fused = torch.cat(
            [
                image_features,
                clinical_features,
                geo_features
            ],
            dim=1
        )

        output = self.classifier(fused)

        return output

In [6]:
model = FusionNet()

print(model)

FusionNet(
  (image_model): CNNModel(
    (backbone): EfficientNet(
      (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNormAct2d(
        32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
        (drop): Identity()
        (act): SiLU(inplace=True)
      )
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (bn1): BatchNormAct2d(
              32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
              (drop): Identity()
              (act): SiLU(inplace=True)
            )
            (aa): Identity()
            (se): SqueezeExcite(
              (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (act1): SiLU(inplace=True)
              (conv_expand): Conv2d(8, 32, kernel_siz

In [7]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("="*60)
print("Total Parameters :",total_params)
print("Trainable :",trainable_params)
print("="*60)

Total Parameters : 4760326
Trainable : 4760326


In [8]:
image = torch.randn(8,3,224,224)

clinical = torch.randn(8,5)

geo = torch.randint(0,2,(8,1)).float()

In [9]:
output = model(
    image,
    clinical,
    geo
)

In [10]:
print("="*60)
print("FusionNet Loaded Successfully")
print("="*60)

print("Image Input :",image.shape)
print("Clinical :",clinical.shape)
print("Geo :",geo.shape)
print("Output :",output.shape)

FusionNet Loaded Successfully
Image Input : torch.Size([8, 3, 224, 224])
Clinical : torch.Size([8, 5])
Geo : torch.Size([8, 1])
Output : torch.Size([8, 2])


In [11]:
probabilities = torch.softmax(output,dim=1)

predictions = torch.argmax(output,dim=1)

print(probabilities)

print(predictions)

tensor([[0.5016, 0.4984],
        [0.4993, 0.5007],
        [0.5009, 0.4991],
        [0.4871, 0.5129],
        [0.5031, 0.4969],
        [0.4934, 0.5066],
        [0.4860, 0.5140],
        [0.5158, 0.4842]], grad_fn=<SoftmaxBackward0>)
tensor([0, 1, 0, 1, 0, 1, 1, 0])


In [12]:
print("="*60)
print("Notebook 09 Completed Successfully")
print("="*60)

print("✔ CNN Features       : 128")
print("✔ Clinical Features  : 16")
print("✔ Geo Features       : 8")
print("✔ Total Features     : 152")
print("✔ Output Classes     : 2")
print("✔ FusionNet Ready")

Notebook 09 Completed Successfully
✔ CNN Features       : 128
✔ Clinical Features  : 16
✔ Geo Features       : 8
✔ Total Features     : 152
✔ Output Classes     : 2
✔ FusionNet Ready
